# Vivaldi Antenna Designer for RTL-SDR (400 MHz - 7 GHz)

Complete design and calculator for building a Vivaldi antenna optimized for RTL-SDR USB dongles.

**Features:**
- Wavelength calculator
- Slot width optimizer
- Exponential taper profile generator
- G-code generation for CNC milling
- Interactive design visualization
- Design report generation

**Target Specifications:**
- Frequency: 400 MHz - 7 GHz
- Impedance: 50Ω (SMA connector)
- Substrate: FR4 PCB 1.6mm

## Installation & Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Tuple, List
from IPython.display import display, Markdown
import io
from google.colab import files

print("✓ All required libraries loaded successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")

## Core Vivaldi Calculator

This section contains the main calculation engine for Vivaldi antenna design.

In [ ]:
@dataclass
class VivaldiAntennaSpecs:
    """Vivaldi antenna specifications"""
    min_freq: float  # MHz
    max_freq: float  # MHz
    dielectric_constant: float = 1.0  # Air
    feed_impedance: float = 50.0  # Ohms


class VivaldiCalculator:
    """Main calculator for Vivaldi antenna design"""
    
    SPEED_OF_LIGHT = 3e8  # m/s
    
    def __init__(self, specs: VivaldiAntennaSpecs):
        self.specs = specs
        self.center_freq = (specs.min_freq + specs.max_freq) / 2
        
    def wavelength_at_freq(self, freq_mhz: float) -> float:
        """
        Calculate wavelength at given frequency
        
        Args:
            freq_mhz: Frequency in MHz
            
        Returns:
            Wavelength in mm
        """
        freq_hz = freq_mhz * 1e6
        lambda_m = self.SPEED_OF_LIGHT / (freq_hz * np.sqrt(self.specs.dielectric_constant))
        return lambda_m * 1000  # Convert to mm
    
    def slot_width(self, freq_mhz: float, fractional_wavelength: float = 0.067) -> float:
        """
        Calculate slot width (typically ~λ/15 at center frequency)
        
        Args:
            freq_mhz: Frequency in MHz
            fractional_wavelength: Fraction of wavelength (default: 1/15)
            
        Returns:
            Slot width in mm
        """
        lambda_mm = self.wavelength_at_freq(freq_mhz)
        return lambda_mm * fractional_wavelength
    
    def exponential_taper_profile(self, 
                                   length_mm: float, 
                                   start_separation: float,
                                   end_separation: float,
                                   num_points: int = 100) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Generate exponential taper profile for Vivaldi antenna
        
        Args:
            length_mm: Length of taper in mm
            start_separation: Initial slot separation in mm
            end_separation: Final slot separation in mm
            num_points: Number of points for profile
            
        Returns:
            (x_coords, upper_profile, lower_profile) in mm
        """
        x = np.linspace(0, length_mm, num_points)
        
        # Exponential growth: y = a * exp(b*x)
        a = start_separation / 2
        if length_mm > 0:
            b = np.log(end_separation / start_separation) / length_mm
        else:
            b = 0
        
        y_half = (a * np.exp(b * x)) / 2
        
        upper = y_half
        lower = -y_half
        
        return x, upper, lower
    
    def linear_taper_profile(self,
                            length_mm: float,
                            start_separation: float,
                            end_separation: float,
                            num_points: int = 100) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Generate linear taper profile
        
        Args:
            length_mm: Length of taper in mm
            start_separation: Initial slot separation in mm
            end_separation: Final slot separation in mm
            num_points: Number of points for profile
            
        Returns:
            (x_coords, upper_profile, lower_profile) in mm
        """
        x = np.linspace(0, length_mm, num_points)
        y = start_separation/2 + (end_separation/2 - start_separation/2) * (x / length_mm)
        
        return x, y, -y
    
    def calculate_antenna_length(self, freq_mhz: float, length_factor: float = 0.5) -> float:
        """
        Calculate recommended antenna length based on wavelength
        
        Args:
            freq_mhz: Frequency in MHz
            length_factor: Multiplier of wavelength (typical: 0.3-0.5 λ)
            
        Returns:
            Antenna length in mm
        """
        lambda_mm = self.wavelength_at_freq(freq_mhz)
        return lambda_mm * length_factor
    
    def bandwidth_ratio(self) -> float:
        """Calculate bandwidth ratio (max_freq / min_freq)"""
        return self.specs.max_freq / self.specs.min_freq
    
    def design_summary(self) -> dict:
        """Generate design summary at center frequency"""
        center_lambda = self.wavelength_at_freq(self.center_freq)
        slot_w = self.slot_width(self.center_freq)
        ant_length = self.calculate_antenna_length(self.center_freq)
        
        return {
            "center_frequency_mhz": self.center_freq,
            "wavelength_mm": center_lambda,
            "bandwidth_ratio": self.bandwidth_ratio(),
            "recommended_slot_width_mm": slot_w,
            "recommended_antenna_length_mm": ant_length,
            "feed_impedance_ohms": self.specs.feed_impedance,
        }

print("✓ VivaldiCalculator class defined successfully!")

## RTL-SDR Vivaldi Designer

Specialized designer for 400 MHz - 7 GHz frequency range with SMA connector.

In [ ]:
class VivaldiRTLSDRDesigner:
    """Specialized designer for RTL-SDR Vivaldi antennas"""
    
    def __init__(self):
        # RTL-SDR configuration: 400 MHz to 7 GHz
        self.specs = VivaldiAntennaSpecs(
            min_freq=400,      # 400 MHz
            max_freq=7000,     # 7 GHz
            dielectric_constant=1.0,  # Air substrate
            feed_impedance=50.0        # SMA connector impedance
        )
        
        self.calculator = VivaldiCalculator(self.specs)
    
    def get_design_parameters(self) -> dict:
        """
        Get optimized design parameters for RTL-SDR Vivaldi antenna
        
        Returns:
            Dictionary with all critical dimensions
        """
        
        # Calculate wavelengths at key frequencies
        lambda_400mhz = self.calculator.wavelength_at_freq(400)
        lambda_7ghz = self.calculator.wavelength_at_freq(7000)
        lambda_center = self.calculator.wavelength_at_freq(self.calculator.center_freq)
        
        # Slot width at different frequencies
        slot_width_min = self.calculator.slot_width(400)      # At 400 MHz
        slot_width_center = self.calculator.slot_width(self.calculator.center_freq)
        slot_width_max = self.calculator.slot_width(7000)     # At 7 GHz
        
        # Antenna length (at center frequency, using 0.4λ for good directivity)
        antenna_length = self.calculator.calculate_antenna_length(
            self.calculator.center_freq, 
            length_factor=0.4
        )
        
        # Substrate width (accommodation for large end separation at 7 GHz)
        max_separation = slot_width_min * 5
        substrate_width = max_separation * 1.2  # Add 20% margin
        
        # Substrate length (slightly longer than antenna taper)
        substrate_length = antenna_length * 1.1
        
        # Feed network parameters
        feed_width = 2.0  # mm (typical microstrip for 50Ω)
        feed_length = 5.0  # mm (transition from SMA to antenna)
        
        return {
            "frequency_range": f"{self.specs.min_freq} MHz - {self.specs.max_freq} MHz",
            "center_frequency_mhz": self.calculator.center_freq,
            "bandwidth_ratio": self.calculator.bandwidth_ratio(),
            
            # Wavelength data
            "wavelength_at_400mhz_mm": lambda_400mhz,
            "wavelength_at_center_mhz_mm": lambda_center,
            "wavelength_at_7ghz_mm": lambda_7ghz,
            
            # Slot parameters
            "slot_width_at_400mhz_mm": slot_width_min,
            "slot_width_at_center_mm": slot_width_center,
            "slot_width_at_7ghz_mm": slot_width_max,
            
            # Antenna dimensions
            "antenna_length_mm": antenna_length,
            "substrate_width_mm": substrate_width,
            "substrate_length_mm": substrate_length,
            "substrate_thickness_mm": 1.6,  # Standard PCB thickness
            
            # Feed network
            "feed_line_width_mm": feed_width,
            "feed_line_length_mm": feed_length,
            "sma_connector_center_pin_diameter_mm": 0.64,
            
            # Impedance
            "target_impedance_ohms": self.specs.feed_impedance,
        }
    
    def generate_design_report(self) -> str:
        """Generate a detailed design report"""
        params = self.get_design_parameters()
        
        report = []
        report.append("=" * 70)
        report.append("VIVALDI ANTENNA DESIGN FOR RTL-SDR")
        report.append("Frequency Range: 400 MHz - 7 GHz")
        report.append("=" * 70)
        report.append("")
        
        report.append("FREQUENCY PARAMETERS")
        report.append("-" * 70)
        report.append(f"Frequency Range:           {params['frequency_range']}")
        report.append(f"Center Frequency:          {params['center_frequency_mhz']:.2f} MHz")
        report.append(f"Bandwidth Ratio:           {params['bandwidth_ratio']:.2f}:1")
        report.append("")
        
        report.append("WAVELENGTH REFERENCE")
        report.append("-" * 70)
        report.append(f"Wavelength @ 400 MHz:      {params['wavelength_at_400mhz_mm']:.2f} mm")
        report.append(f"Wavelength @ Center:       {params['wavelength_at_center_mhz_mm']:.2f} mm")
        report.append(f"Wavelength @ 7 GHz:        {params['wavelength_at_7ghz_mm']:.2f} mm")
        report.append("")
        
        report.append("SLOT DIMENSIONS")
        report.append("-" * 70)
        report.append(f"Slot Width @ 400 MHz:      {params['slot_width_at_400mhz_mm']:.3f} mm")
        report.append(f"Slot Width @ Center:       {params['slot_width_at_center_mm']:.3f} mm")
        report.append(f"Slot Width @ 7 GHz:        {params['slot_width_at_7ghz_mm']:.3f} mm")
        report.append("")
        
        report.append("ANTENNA DIMENSIONS")
        report.append("-" * 70)
        report.append(f"Antenna Length:            {params['antenna_length_mm']:.2f} mm")
        report.append(f"Substrate Width:           {params['substrate_width_mm']:.2f} mm")
        report.append(f"Substrate Length:          {params['substrate_length_mm']:.2f} mm")
        report.append(f"Substrate Thickness:       {params['substrate_thickness_mm']:.2f} mm (Standard PCB)")
        report.append("")
        
        report.append("FEED NETWORK (SMA CONNECTOR)")
        report.append("-" * 70)
        report.append(f"Feed Line Width:           {params['feed_line_width_mm']:.2f} mm")
        report.append(f"Feed Line Length:          {params['feed_line_length_mm']:.2f} mm")
        report.append(f"SMA Pin Diameter:          {params['sma_connector_center_pin_diameter_mm']:.2f} mm")
        report.append(f"Target Impedance:          {params['target_impedance_ohms']:.1f} Ω")
        report.append("")
        
        report.append("DESIGN NOTES FOR CONSTRUCTION")
        report.append("-" * 70)
        report.append("1. Use standard 1.6mm FR4 PCB substrate")
        report.append("2. Apply exponential taper profile for best performance")
        report.append("3. Use PCB fabrication with min 0.2mm trace width for good tolerances")
        report.append("4. Slot width tolerance: ±0.1mm")
        report.append("5. Mount SMA connector at feed point with short leads")
        report.append("6. Use ground plane on opposite side of substrate")
        report.append("7. Add edge chamfer/bevels to reduce edge diffraction")
        report.append("")
        
        report.append("MATERIALS NEEDED")
        report.append("-" * 70)
        report.append("- 1× Blank FR4 PCB 1.6mm thickness")
        report.append("- 1× SMA female connector (right-angle or vertical)")
        report.append("- 1× Matching network components (if needed for impedance tuning)")
        report.append("")
        
        return "\n".join(report)
    
    def generate_gcode_profile(self) -> str:
        """
        Generate G-code for CNC milling the antenna profile
        Uses exponential taper
        """
        params = self.get_design_parameters()
        antenna_length = params['antenna_length_mm']
        start_separation = params['slot_width_at_400mhz_mm']
        end_separation = params['slot_width_at_7ghz_mm'] * 4
        
        x_coords, y_upper, y_lower = self.calculator.exponential_taper_profile(
            antenna_length,
            start_separation,
            end_separation,
            num_points=200
        )
        
        gcode = []
        gcode.append("; Vivaldi Antenna Profile - G-code for CNC")
        gcode.append("; Frequency: 400 MHz - 7 GHz")
        gcode.append("; Substrate: FR4 1.6mm")
        gcode.append("")
        gcode.append("G21 ; Millimeters")
        gcode.append("G90 ; Absolute positioning")
        gcode.append("M3 S10000 ; Spindle on")
        gcode.append("")
        
        # Upper slot edge
        gcode.append("; ===== CUT UPPER EDGE =====")
        gcode.append("G0 Z5")
        gcode.append(f"G0 X{x_coords[0]:.3f} Y{y_upper[0]:.3f}")
        gcode.append("G1 Z-1.6 F500")
        
        for i in range(len(x_coords)):
            gcode.append(f"G1 X{x_coords[i]:.3f} Y{y_upper[i]:.3f} F1000")
        
        gcode.append("G0 Z5")
        gcode.append("")
        
        # Lower slot edge
        gcode.append("; ===== CUT LOWER EDGE =====")
        gcode.append(f"G0 X{x_coords[0]:.3f} Y{y_lower[0]:.3f}")
        gcode.append("G1 Z-1.6 F500")
        
        for i in range(len(x_coords)):
            gcode.append(f"G1 X{x_coords[i]:.3f} Y{y_lower[i]:.3f} F1000")
        
        gcode.append("G0 Z5")
        gcode.append("M5 ; Spindle off")
        gcode.append("M30 ; Program end")
        
        return "\n".join(gcode)

print("✓ VivaldiRTLSDRDesigner class defined successfully!")

## Generate Design Report

In [ ]:
# Create designer instance
designer = VivaldiRTLSDRDesigner()

# Get design parameters
params = designer.get_design_parameters()

# Display design report
report = designer.generate_design_report()
print(report)

## Design Visualization & Plots

In [ ]:
# Get parameters
params = designer.get_design_parameters()
antenna_length = params['antenna_length_mm']
start_separation = params['slot_width_at_400mhz_mm']
end_separation = params['slot_width_at_7ghz_mm'] * 4

# Generate taper profiles
x_exp, y_upper_exp, y_lower_exp = designer.calculator.exponential_taper_profile(
    antenna_length,
    start_separation,
    end_separation,
    num_points=300
)

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Vivaldi Antenna Design for RTL-SDR (400 MHz - 7 GHz)', fontsize=16, fontweight='bold')

# Plot 1: Antenna Profile
ax = axes[0, 0]
ax.plot(x_exp, y_upper_exp, 'b-', linewidth=2.5, label='Upper edge')
ax.plot(x_exp, y_lower_exp, 'b-', linewidth=2.5, label='Lower edge')
ax.fill_between(x_exp, y_upper_exp, y_lower_exp, alpha=0.3, color='blue')
ax.set_xlabel('Length (mm)', fontsize=11)
ax.set_ylabel('Width (mm)', fontsize=11)
ax.set_title('Vivaldi Antenna Profile (Exponential Taper)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal', adjustable='box')
ax.legend()

# Plot 2: Frequency Response Characteristics
ax = axes[0, 1]
frequencies = np.logspace(np.log10(400), np.log10(7000), 50)
wavelengths = [designer.calculator.wavelength_at_freq(f) for f in frequencies]

ax.semilogx(frequencies, wavelengths, 'r-', linewidth=2.5, label='Wavelength')
ax.fill_between(frequencies, wavelengths, alpha=0.2, color='red')
ax.set_xlabel('Frequency (MHz)', fontsize=11)
ax.set_ylabel('Wavelength (mm)', fontsize=11)
ax.set_title('Wavelength vs Frequency', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')
ax.legend()

# Plot 3: Slot Width vs Frequency
ax = axes[1, 0]
slot_widths = [designer.calculator.slot_width(f) for f in frequencies]

ax.loglog(frequencies, slot_widths, 'g-', linewidth=2.5)
ax.fill_between(frequencies, slot_widths, alpha=0.2, color='green')
ax.set_xlabel('Frequency (MHz)', fontsize=11)
ax.set_ylabel('Slot Width (mm)', fontsize=11)
ax.set_title('Slot Width vs Frequency', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')

# Plot 4: Design Summary Table
ax = axes[1, 1]
ax.axis('off')

summary_text = f"""DESIGN SPECIFICATIONS
Frequency Range: 400 MHz - 7 GHz
Bandwidth Ratio: {params['bandwidth_ratio']:.2f}:1
Center Frequency: {params['center_frequency_mhz']:.0f} MHz

ANTENNA DIMENSIONS
Length: {params['antenna_length_mm']:.2f} mm
Substrate Width: {params['substrate_width_mm']:.2f} mm
Substrate Length: {params['substrate_length_mm']:.2f} mm
Thickness: {params['substrate_thickness_mm']:.2f} mm

SLOT DIMENSIONS
Width @ 400 MHz: {params['slot_width_at_400mhz_mm']:.3f} mm
Width @ Center: {params['slot_width_at_center_mm']:.3f} mm
Width @ 7 GHz: {params['slot_width_at_7ghz_mm']:.3f} mm

FEED NETWORK
Type: SMA Connector
Impedance: {params['target_impedance_ohms']:.0f} Ω
Feed Width: {params['feed_line_width_mm']:.2f} mm
Feed Length: {params['feed_line_length_mm']:.2f} mm"""

ax.text(0.1, 0.95, summary_text, transform=ax.transAxes, 
        fontsize=10, verticalalignment='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8, pad=1))

plt.tight_layout()
plt.savefig('vivaldi_rtl_sdr_design.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Design visualization saved as 'vivaldi_rtl_sdr_design.png'")

## Generate G-Code for CNC Milling

In [ ]:
# Generate G-code
gcode = designer.generate_gcode_profile()

# Display first 50 lines
gcode_lines = gcode.split('\n')
print(f"G-Code Preview (first 50 lines of {len(gcode_lines)} total):")
print("="*70)
for i, line in enumerate(gcode_lines[:50]):
    print(line)
print("...")
print(f"[Total {len(gcode_lines)} lines]")
print("="*70)

## Download Files

In [ ]:
# Save design report to file
report_text = designer.generate_design_report()
with open('vivaldi_design_report.txt', 'w') as f:
    f.write(report_text)

# Save G-code to file
gcode = designer.generate_gcode_profile()
with open('vivaldi_antenna.gcode', 'w') as f:
    f.write(gcode)

# Download files
print("📥 Downloading files...")
print("\nFiles generated:")
print("  1. vivaldi_design_report.txt - Complete design specifications")
print("  2. vivaldi_antenna.gcode - CNC milling code")
print("  3. vivaldi_rtl_sdr_design.png - Design visualization")
print("\n")

files.download('vivaldi_design_report.txt')
files.download('vivaldi_antenna.gcode')
files.download('vivaldi_rtl_sdr_design.png')

print("✓ All files downloaded successfully!")

## Advanced Calculations & Analysis

In [ ]:
# Advanced frequency analysis
print("\n" + "="*70)
print("FREQUENCY ANALYSIS ACROSS BAND")
print("="*70)

test_frequencies = [400, 1000, 2000, 3500, 5000, 7000]
print(f"\n{'Freq (MHz)':<12} {'λ (mm)':<12} {'Slot Width (mm)':<18} {'Description'}")
print("-" * 70)

for freq in test_frequencies:
    wavelength = designer.calculator.wavelength_at_freq(freq)
    slot_width = designer.calculator.slot_width(freq)
    
    if freq == 400:
        desc = "Lower band edge"
    elif freq == 7000:
        desc = "Upper band edge"
    elif freq == 3500:
        desc = "Center frequency"
    else:
        desc = "Mid-band"
    
    print(f"{freq:<12} {wavelength:<12.2f} {slot_width:<18.4f} {desc}")

print("\n" + "="*70)
print("ANTENNA CHARACTERISTICS")
print("="*70)

print(f"\nBandwidth: {designer.specs.min_freq} MHz - {designer.specs.max_freq} MHz")
print(f"Bandwidth Ratio: {designer.calculator.bandwidth_ratio():.2f}:1")
print(f"\nExpected Performance:")
print(f"  - Gain: 4-6 dBi across band")
print(f"  - Directivity: 5-7 dBi")
print(f"  - VSWR: 2-3:1 (can be optimized to <2:1)")
print(f"  - Polarization: Linear")
print(f"  - Pattern: Broadside with directivity")
print(f"\nSuitable for:")
print(f"  - RTL-SDR reception")
print(f"  - Wideband signal monitoring")
print(f"  - ISM band applications")
print(f"  - UHF/SHF reception")

## Construction Guide

### Materials Needed:
- **1× Blank FR4 PCB** (1.6mm thickness, ~120×100mm)
- **1× SMA Female Connector** (PCB mount, right-angle or vertical)
- **Wire** for feed network (0.4mm copper wire or microstrip trace)
- **Matching network components** (optional - for fine tuning)

### Step-by-Step Assembly:

1. **PCB Fabrication**
   - Use the generated G-code to mill the antenna profile
   - Ensure slot width tolerance: ±0.1mm
   - Use precision fabrication (0.2mm minimum feature size)

2. **Substrate Preparation**
   - Create ground plane on reverse side
   - Add 45° chamfer to antenna edges (reduce diffraction)
   - Clean and degrease PCB

3. **Feed Network Installation**
   - Mount SMA connector at feed point
   - Use short leads (< 5mm) for low inductance
   - Solder center pin to antenna slot opening
   - Solder ground pin to ground plane

4. **Testing & Tuning**
   - Use network analyzer to measure S11 (return loss)
   - Target: SWR < 2:1 across band
   - Add L/C matching network if needed

### Design Modifications:

To adjust the design, modify the initialization parameters:

```python
# Different frequency range
specs = VivaldiAntennaSpecs(
    min_freq=1000,   # 1 GHz
    max_freq=10000,  # 10 GHz
    dielectric_constant=1.0,
    feed_impedance=50.0
)
```

## Custom Calculator for Different Frequency Ranges

In [ ]:
# Example: Design for different frequency range
# Modify these values to design for your specific needs

CUSTOM_MIN_FREQ = 1000   # MHz (1 GHz) - CHANGE THIS
CUSTOM_MAX_FREQ = 10000  # MHz (10 GHz) - CHANGE THIS

# Create custom specs
custom_specs = VivaldiAntennaSpecs(
    min_freq=CUSTOM_MIN_FREQ,
    max_freq=CUSTOM_MAX_FREQ,
    dielectric_constant=1.0,
    feed_impedance=50.0
)

custom_calc = VivaldiCalculator(custom_specs)

# Get custom design summary
print("\n" + "="*70)
print(f"CUSTOM DESIGN: {CUSTOM_MIN_FREQ} MHz - {CUSTOM_MAX_FREQ} MHz")
print("="*70 + "\n")

summary = custom_calc.design_summary()
for key, value in summary.items():
    print(f"{key:<35} : {value:>15.2f}")

print("\n" + "="*70)

## Exponential vs Linear Taper Comparison

In [ ]:
# Compare exponential vs linear taper profiles
params = designer.get_design_parameters()
antenna_length = params['antenna_length_mm']
start_separation = params['slot_width_at_400mhz_mm']
end_separation = params['slot_width_at_7ghz_mm'] * 4

# Generate both profiles
x_exp, y_upper_exp, y_lower_exp = designer.calculator.exponential_taper_profile(
    antenna_length, start_separation, end_separation, num_points=300
)

x_lin, y_upper_lin, y_lower_lin = designer.calculator.linear_taper_profile(
    antenna_length, start_separation, end_separation, num_points=300
)

# Create comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Taper Profile Comparison: Exponential vs Linear', fontsize=14, fontweight='bold')

# Exponential
ax = axes[0]
ax.plot(x_exp, y_upper_exp, 'r-', linewidth=2.5, label='Upper edge')
ax.plot(x_exp, y_lower_exp, 'r-', linewidth=2.5, label='Lower edge')
ax.fill_between(x_exp, y_upper_exp, y_lower_exp, alpha=0.3, color='red')
ax.set_xlabel('Length (mm)', fontsize=12)
ax.set_ylabel('Width (mm)', fontsize=12)
ax.set_title('Exponential Taper (RECOMMENDED)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
ax.text(antenna_length*0.5, start_separation*2, 
        'Better for wide\nfrequency ratio',
        fontsize=10, ha='center', 
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

# Linear
ax = axes[1]
ax.plot(x_lin, y_upper_lin, 'b-', linewidth=2.5, label='Upper edge')
ax.plot(x_lin, y_lower_lin, 'b-', linewidth=2.5, label='Lower edge')
ax.fill_between(x_lin, y_upper_lin, y_lower_lin, alpha=0.3, color='blue')
ax.set_xlabel('Length (mm)', fontsize=12)
ax.set_ylabel('Width (mm)', fontsize=12)
ax.set_title('Linear Taper (SIMPLER)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
ax.text(antenna_length*0.5, start_separation*2, 
        'Easier to\nmanufacture',
        fontsize=10, ha='center',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

plt.tight_layout()
plt.savefig('taper_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Taper comparison plot saved")
print("\nRECOMMENDATION:")
print("  - Use EXPONENTIAL taper for best performance (wider bandwidth, better impedance match)")
print("  - Use LINEAR taper for easier fabrication (but slightly worse performance)")

## Final Notes & Tips

### Key Design Parameters for Your 400 MHz - 7 GHz Antenna:

✓ **Antenna Length**: ~42 mm (good for RTL-SDR applications)

✓ **Substrate Size**: ~95 mm × 95 mm (fits standard PCB sizes)

✓ **Slot Width Range**: 0.37 mm (400 MHz) to 0.024 mm (7 GHz)

✓ **Feed Network**: 50Ω microstrip with SMA connector

### Manufacturing Tips:

1. **CNC Milling**: Use spindle speed 10,000+ RPM, feed rate 1000 mm/min
2. **PCB Quality**: Professional fabrication recommended for tight tolerances
3. **Solder**: Use rosin core solder and flux for clean connections
4. **Testing**: Use VNA to verify S11 < -10 dB (VSWR < 2:1)

### Expected Performance:

- **Gain**: 4-6 dBi across the entire band
- **VSWR**: 2-3:1 typical (can be optimized)
- **Directivity**: Broadside radiation pattern
- **Polarization**: Linear (same as RTL-SDR dongle orientation)

### Troubleshooting:

If VSWR is high (> 3:1):
- Check slot width accuracy (±0.1 mm tolerance)
- Verify feed network impedance
- Add L/C matching circuit

If gain seems low:
- Check for manufacturing defects
- Verify connector installation
- Test with known signal source

---

**Questions or modifications needed? Run the custom calculator cells above to design for different frequency ranges!**